# Optical Flow

Optical Flow is technique used to extract motion from multiple images, in practice we usually use only two frames of a video.

With Optical Flow, we compute a motion vector $(u, v)$ which encodes de displacement in $x$ and $y$ directions betwween two consecutive frames.

In [76]:
!pip install git+https://github.com/Guillem96/spynet-pytorch

  Cloning https://github.com/Guillem96/spynet-pytorch to /tmp/pip-req-build-uabqtaex
  Running command git clone --filter=blob:none --quiet https://github.com/Guillem96/spynet-pytorch /tmp/pip-req-build-uabqtaex
  Resolved https://github.com/Guillem96/spynet-pytorch to commit 3685b030b24e33379a09c3b8dcaf4974843e8e67
  Preparing metadata (setup.py) ... done


In [77]:
import os
# Use a reliable direct link to the taxi sequence
!rm -rf taxi taxi.zip
!wget -O taxi.zip https://raw.githubusercontent.com/HarisIqbal88/OpticalFlow/master/taxi.zip
!mkdir -p taxi
!unzip -q taxi.zip -d taxi
!rm taxi.zip

# The zip usually contains a folder named 'taxi', let's move files to root 'taxi' if needed
if os.path.exists('taxi/taxi'):
    !mv taxi/taxi/* taxi/
    !rm -rf taxi/taxi

print("Download and extraction complete.")

--2026-04-12 02:39:56--  https://raw.githubusercontent.com/HarisIqbal88/OpticalFlow/master/taxi.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-12 02:39:56 ERROR 404: Not Found.

[taxi.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of taxi.zip or
        taxi.zip.zip, and cannot find taxi.zip.ZIP, period.
Download and extraction complete.


In [78]:
from pathlib import Path

import torch
import cv2 as cv
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

import torchvision.transforms as T

import spynet

## Formulation

Optical flow is based on the Brightness consistency assumtion. The intensity in location $(x, y)$ at time $t$ is almost equal at time $t + 1$.

$ f(x, y, t) = f(x + dx, y + dy, t + 1) $

$(dx, dy) = (u, v)$

With taylor series we can express the equation above as follows

$ f_x u + f_y v + f_t = 0 $

Above equation is called Optical Flow equation. In it, we can find $f_x$ and $f_y$, they are image gradients. Similarly $f_t$ is the gradient along time. But $(u,v$) is unknown. We cannot solve this one equation with two unknown variables. So several methods are provided to solve this problem:

- Discrete optimization methods
- Differential methods such as: [Lucas-kanade](http://cseweb.ucsd.edu/classes/sp02/cse252/lucaskanade81.pdf) and [Horn–Schunck](http://dspace.mit.edu/handle/1721.1/6337)
- *New* Deep Learning approaches like FlowNet and SpyNet


## Hamburg taxi sequence

Famous image sequence. Old days, around 1970, generating fast sequences of images such as videos and move them to the computer was difficult. Two researchers installed a camera on their office window, and they were continously taking photos of the street.

In [79]:
from pathlib import Path
# Filter for images and ensure we are looking in the right place
taxis_fnames = sorted([str(p) for p in Path('taxi').rglob('*') if p.suffix.lower() in ['.pgm', '.jpg', '.png']])
print(f'Number of frames found: {len(taxis_fnames)}')
if len(taxis_fnames) > 0:
    print(f'Sample path: {taxis_fnames[0]}')

Number of frames found: 0


In [80]:
STRIDE = 5 #@param {type: "slider", min: 1, max: 5}

In [81]:
import numpy as np
from PIL import Image

# Added a check to prevent the ValueError if the list is empty
if len(taxis_fnames) > STRIDE:
    rand_idx = np.random.randint(0, len(taxis_fnames) - STRIDE)
    taxi1 = Image.open(taxis_fnames[rand_idx])
    taxi2 = Image.open(taxis_fnames[rand_idx + STRIDE])
    print(f"Selected frames {rand_idx} and {rand_idx + STRIDE}")
else:
    print("Error: Not enough frames found in the taxi directory.")

Error: Not enough frames found in the taxi directory.


In [82]:
if 'taxi1' in locals():
    plt.subplot(121)
    plt.imshow(taxi1, cmap='gray')
    plt.axis('off')

    plt.subplot(122)
    plt.imshow(taxi2, cmap='gray')
    plt.axis('off')
    plt.show()
else:
    print("Error: 'taxi1' is not defined. Please ensure the download and frame selection cells ran successfully.")

Error: 'taxi1' is not defined. Please ensure the download and frame selection cells ran successfully.


OpenCV provides an algorithm to find the optical flow. It computes the optical flow for all the points in the frame. It is based on Gunner Farneback’s algorithm which is explained in [Two-Frame Motion Estimation Based on Polynomial Expansion](https://www.diva-portal.org/smash/get/diva2:273847/FULLTEXT01.pdf).

In [83]:
if 'taxi1' in locals() and 'taxi2' in locals():
    flow = cv.calcOpticalFlowFarneback(np.array(taxi1),
                                       np.array(taxi2),
                                       None, 0.5, 3, 15, 3, 5, 1.2, 0)
    print("Optical flow calculated successfully.")
else:
    print("Error: 'taxi1' or 'taxi2' is not defined. Please run the frame selection cell (iy_5BtGXUiDP) first.")

Error: 'taxi1' or 'taxi2' is not defined. Please run the frame selection cell (iy_5BtGXUiDP) first.


We can display the displacement of each pixel using a quiver plot.

In [84]:
if 'taxi1' in locals() and 'taxi2' in locals() and 'flow' in locals():
    step = 4
    plt.figure(figsize=(20, 5))

    plt.subplot(131)
    plt.title('Frame 1')
    plt.imshow(taxi1, cmap='gray')
    plt.axis('off')

    plt.subplot(132)
    plt.title('Frame 2')
    plt.imshow(taxi2, cmap='gray')
    plt.axis('off')

    plt.subplot(133)
    plt.title('Optical Flow')
    plt.quiver(np.arange(0, 256, step), np.arange(190, -1, -step),
               flow[::step, ::step, 0], flow[::step, ::step, 1])
    plt.axis('off')
    plt.show()
else:
    print("Error: 'taxi1', 'taxi2', or 'flow' is not defined. Please run the download, selection, and flow calculation cells first.")

Error: 'taxi1', 'taxi2', or 'flow' is not defined. Please run the download, selection, and flow calculation cells first.


Also, we can get a prettier representation of Optical Flow mapping the vector magnitudes to a color:

![](https://www.researchgate.net/profile/Christophoros_Nikou/publication/266149545/figure/fig1/AS:392088710598656@1470492641144/The-optical-flow-field-color-coding-Smaller-vectors-are-lighter-and-color-represents-the.png)

Where the vertical axis corresponds to the $v$ and the horizontal axis to $u$

In [85]:
if 'flow' in locals():
    plt.title('Colored optical flow')
    plt.imshow(spynet.flow.flow_to_image(flow))
    plt.axis('off')
    plt.show()
else:
    print("Error: 'flow' is not defined. Please run the calculation cell (_jY_iXbLUiDU) first.")

Error: 'flow' is not defined. Please run the calculation cell (_jY_iXbLUiDU) first.


In [86]:
all_taxi_images = [np.array(Image.open(o)) for o in taxis_fnames]
optical_flows = [cv.calcOpticalFlowFarneback(im1, im2,
                  None, 0.5, 3, 15, 3, 5, 1.2, 0)
                 for im1, im2 in zip(all_taxi_images[:-1], all_taxi_images[1:])]
optical_flows = [spynet.flow.flow_to_image(o) for o in optical_flows]

In [87]:
def display_video(images):
    if not images:
        print("Error: The image list is empty. Cannot display video.")
        return

    fig = plt.figure()
    plt.axis('off')
    im = plt.imshow(images[0], cmap='gray')
    plt.close()

    def init():
        im.set_data(images[0])

    def animate(i):
        im.set_data(images[i])
        return im

    anim = animation.FuncAnimation(
        fig, animate, init_func=init, frames=len(images), interval=50)
    display(HTML(anim.to_html5_video()))

if len(optical_flows) > 0:
    display_video(optical_flows)
if len(all_taxi_images) > 0:
    display_video(all_taxi_images)
else:
    print("No images found to display. Please check the data loading steps.")

No images found to display. Please check the data loading steps.


## Deep Learning Approach

To compute the optical flow with deep learning, we are going to use the *coarse to fine grained* approach defined in the SpyNet paper. Another famous NN to compute the Optical Flow is called FlowNet, but this net is much larger than the one we are using in the cells below: Spatial Pyramid Network (SPyNet) is much simpler and 96% smaller than FlowNet in terms of model parameters

In [89]:
import torch
import functools
import os

import requests
from pathlib import Path
import spynet

# 1. Force a fresh download of weights to avoid corruption
weights_url = "https://github.com/Guillem96/spynet-pytorch/releases/download/v1.0/spynet_flying_chairs-c31fde10.pth"
dst_path = Path("spynet_flying_chairs-c31fde10.pth")

print("Downloading weights...")
r = requests.get(weights_url, allow_redirects=True)
with open(dst_path, 'wb') as f:
    f.write(r.content)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

@torch.no_grad()
def predict(frames1, frames2):
    images1 = torch.stack([tfms(o.convert('RGB')) for o in frames1]).to(device)
    images2 = torch.stack([tfms(o.convert('RGB')) for o in frames2]).to(device)
    return model((images1, images2))

tfms = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[.485, .406, .456],
                std= [.229, .225, .224])
])

# 2. Manual model initialization to bypass library's loader issues
try:
    # Initialize empty model
    model = spynet.SpyNet()

    # Load state dict manually with weights_only=False for compatibility
    checkpoint = torch.load(dst_path, map_location=device, weights_only=False)

    # The checkpoint might be the state_dict itself or contain it
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)

    model.to(device)
    model.eval()
    print("Model loaded successfully manually.")
except Exception as e:
    print(f"Failed to load model: {e}")

if 'taxi1' in locals() and 'taxi2' in locals():
    flow = predict([taxi1], [taxi2])[0]
    plt.imshow(spynet.flow.flow_to_image(flow.cpu()))
    plt.axis('off')
    plt.show()
else:
    print("Error: 'taxi1' or 'taxi2' not found. Please ensure data cells ran successfully.")

Failed to load model: At least one argument (units or k) must bespecified
Error: 'taxi1' or 'taxi2' not found. Please ensure data cells ran successfully.


In [90]:
images = [Image.open(o) for o in taxis_fnames]
all_flows = []
for im1, im2 in zip(images[1:], images[:-1]):
    flow = predict([im1], [im2])[0]
    all_flows.append(spynet.flow.flow_to_image(flow))

display_video(all_flows)

Error: The image list is empty. Cannot display video.


## Optical Flow for action Recognition

Most of the top performing action recognition methods use optical flow as a “black box” input. In some video action recognition classifiers that are based on 2D convolutions, which actually they are not able to extract temporal features, the optical flow is a good external feature that let us improve the classification results in front of the raw RGB images.

While it may seem intuitive to include image motion in a task related to video, often video categories in datasets can be identified from a single image. For instance, playing the guitar can esily be predicted if a guitar appears within the image. Therefore, we can think that optical flow improves the models' results is because it encodes the temporal information that 2D convolutions cannot see.

But surprisingly, in the paper "On the Integration of Optical Flow and Action Recognition" they experiment with optical flow and demonstate that, actually, it is good because of its capability of representing scenes independently of the appearance. Also, the paper describes that as the datasets get even larger, the gap between optical flow and images accuracy decreases because the model sees the same action a lot more environments with brightness, color and objects variations.

**Models that use Optical Flow as input**

- [LRCN](https://arxiv.org/abs/1411.4389)

## References

\[1\] High Accuracy Optical Flow Method Based on a Theory for Warping: Implementation and Qualitative/Quantitative Evaluation - Mohammad Faisal and John Barron

\[2\] On the Integration of Optical Flow and Action Recognition - Laura Sevilla-Lara, Yiyi Liao et.al

\[3\] FlowNet: Learning Optical Flow with Convolutional Networks

\[4\] Optical Flow Estimation using a Spatial Pyramid Network - Anurag Ranjan, Michael J. Black